In [ ]:
#The code in this block comes directly from data_sort_and_split.ipynb instead sclaing is uneccesary as it is not required for Forest models
import pandas as pd
from sklearn.model_selection import train_test_split

heart_data = pd.read_csv("heart.csv")

heart_data['Sex_F'] = heart_data['Sex'].map({'M': 0, 'F': 1})
heart_data['ExerciseAngina'] = heart_data['ExerciseAngina'].map({'N': 0, 'Y': 1})
heart_data = heart_data.drop(columns=['Sex'])

heart_data['ChestPainType'] = pd.Categorical(heart_data['ChestPainType'], categories=['ASY', 'ATA', 'NAP', 'TA'])
heart_data['RestingECG'] = pd.Categorical(heart_data['RestingECG'], categories=['Normal', 'LVH', 'ST'])
heart_data['ST_Slope'] = pd.Categorical(heart_data['ST_Slope'], categories=['Up', 'Flat', 'Down'])

categorical_cols = ['ChestPainType', 'RestingECG', 'ST_Slope']
heart_data = pd.get_dummies(heart_data, columns=categorical_cols, drop_first=True, dtype=int)

feature_matrix = heart_data.drop("HeartDisease", axis=1)
target_labels = heart_data["HeartDisease"]

features_train, features_test, targets_train, targets_test = train_test_split(
    feature_matrix,
    target_labels,
    test_size=0.20,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Why: n_estimators=100 builds 100 individual decision trees and combines their predictions,
# which produces a much more stable and accurate result than a single decision tree alone.
# Why: criterion='entropy' measures the quality of each split using information gain,
# allowing each tree to always choose the most informative question to ask about a patient.
# Why: max_depth=10 limits how deep each tree can grow, preventing individual trees from
# memorizing the training data and failing on new patients (overfitting).
# Why: max_features='sqrt' means each tree only sees a random subset of features at each
# split, forcing the trees to be different from each other and reducing overfitting.
# Why: bootstrap=True means each tree is trained on a random sample of patients with
# replacement, which adds diversity across trees and improves generalization.
# Why: class_weight='balanced' ensures the model pays extra attention to heart disease
# patients since our dataset may have more healthy patients than sick ones.
# Why: random_state=42 ensures the model produces the same results every time it is run,
# making our results reproducible and comparable with our teammates.
random_forest_classifier = RandomForestClassifier(
    n_estimators=100,
    criterion='entropy',
    max_depth=10,
    max_features='sqrt',
    bootstrap=True,
    class_weight='balanced',
    random_state=42
)

# Why: We train the model on 80% of the patients so it can learn patterns without ever
# seeing the test set, preventing data leakage.
random_forest_classifier.fit(features_train, targets_train)

# Why: We test the model on the remaining 20% of patients it has never seen before,
# giving us an honest measure of real-world performance.
predicted_labels = random_forest_classifier.predict(features_test)

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Why: predict_proba gives us the probability of heart disease for each patient rather than
# a hard 0 or 1, which is needed to calculate the ROC-AUC score.
# Why: We take [:, 1] to get only the probability of the positive class (heart disease).
predicted_proba = random_forest_classifier.predict_proba(features_test)[:, 1]

# Why: Accuracy alone can be misleading in medical classification — a model that predicts
# no heart disease for every patient would still score high on accuracy but catch nobody.
accuracy  = accuracy_score(targets_test, predicted_labels)

# Why: Precision measures how reliable our positive predictions are — of all patients we
# flagged as having heart disease, how many actually do.
precision = precision_score(targets_test, predicted_labels)

# Why: Recall measures how well we detect true heart disease cases — of all patients who
# actually have heart disease, how many did we correctly catch. This is our most important
# metric since missing a real heart disease patient is far more dangerous than a false alarm.
recall    = recall_score(targets_test, predicted_labels)

# Why: F1-score combines precision and recall into one measure, giving us a balance between
# catching heart disease patients and avoiding too many false alarms.
f1        = f1_score(targets_test, predicted_labels)

# Why: ROC-AUC uses probabilities rather than hard predictions to measure how well the model
# separates heart disease patients from healthy ones across all possible thresholds.
roc_auc   = roc_auc_score(targets_test, predicted_proba)

print("Random Forest Model Performance:")
print(f"  Accuracy:  {accuracy:.4f}")
print(f"  Precision: {precision:.4f}")
print(f"  Recall:    {recall:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {roc_auc:.4f}")